In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

from numpy import savetxt
from res import RESERVOIRE_SIMPLE

import matplotlib.pyplot as plt
from numpy import linalg as LA

from nilearn import datasets, image, masking
import numpy as np

# Load Schaefer atlas (100-region, 7-network)
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
atlas_img = atlas['maps']
labels = atlas['labels']

input_factor = 0.

if_save_res = False
if_load_res = False

sigma_dynamics = .0

recurrent_factor = .1


In [ ]:
# â”€â”€ Optimised reservoir class (drop-in replacement for res.RESERVOIRE_SIMPLE) â”€â”€
import numpy as np

class RESERVOIRE_SIMPLE:
    """Rate-network reservoir with pre-computed time constants."""

    def __init__(self, par):
        self.N, self.I, self.O, self.T = par['shape']
        self.dt   = par['dt']
        tau_m     = np.linspace(par['tau_m_f'], par['tau_m_s'], self.N)

        # Pre-compute per-unit decay/gain (avoids exp inside the hot loop)
        self.decay = np.exp(-self.dt / tau_m)           # shape (N,)
        self.gain  = 1.0 - self.decay                   # shape (N,)

        self.J    = np.random.normal(0., 1. / np.sqrt(self.N), (self.N, self.N))
        self.Jin  = np.random.normal(0., par['sigma_input'],    (self.N, self.I))
        self.Jout = np.zeros((self.O, self.N))
        self.h_Jout = np.zeros((self.O,))
        self.y    = np.zeros((self.O,))
        self.X    = np.zeros((self.N,))
        self._Xbuf = np.empty((self.N,))   # reusable noise buffer
        self.par  = par

    def step_rate(self, inp, sigma_dyn=0.0):
        if sigma_dyn:
            np.add(self.X, np.random.normal(0., sigma_dyn, self.N), out=self._Xbuf)
            Xd = self._Xbuf
        else:
            Xd = self.X                     # no copy needed; J@Xd is read-only
        self.X = self.decay * self.X + self.gain * (self.J @ Xd + self.Jin @ inp)
        self.y = self.Jout @ self.X + self.h_Jout
        return self.X

    def reset(self, init=None):
        self.X[:] = 0.
        self.y[:] = 0.


In [ ]:
import os
import numpy as np
from scipy.signal import butter, filtfilt

def lowpass_filter_rows(X, fs, cutoff, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low')
    return np.array([filtfilt(b, a, row) for row in X])

fs = 1000
cutoff = 150

timeseries_base = r"C:\Users\user\Desktop\2026.AD_MotionCorrection\timeseries"

T_STANDARD = 140
N_PARCELS  = 121

collected_signals = []
identifiers_list = []

for group, state_id in [("AD", "AD"), ("MCI", "MCI"), ("CN", "CC")]:
    folder = os.path.join(timeseries_base, group)
    if not os.path.isdir(folder):
        continue
    fnames = sorted([f for f in os.listdir(folder) if f.endswith(".npy")])
    for fname in fnames:
        arr = np.load(os.path.join(folder, fname))
        if arr.shape[1] != T_STANDARD:
            print(f"Skipped (T={arr.shape[1]}) {fname}")
            continue
        if arr.shape[0] != N_PARCELS:
            print(f"Skipped (N={arr.shape[0]}) {fname}")
            continue
        arr = arr.T
        sub_id = fname.split("_")[0]
        identifiers_list.append([state_id, sub_id, "Resting_State_fMRI"])
        collected_signals.append(arr)
        print(f"Loaded {fname}, shape={arr.shape}")

identifiers = np.array(identifiers_list, dtype=object)


In [ ]:
patient_ID = [identifiers[k][1] for k in range(len(identifiers))]


In [ ]:
len(identifiers)

In [ ]:
state_ID = [identifiers[k][0] for k in range(len(identifiers))]

In [ ]:
len(state_ID)

In [ ]:
rec_type = [identifiers[k][2] for k in range(len(identifiers))]

In [ ]:
# Filter indices where rec_type is 'Resting_State_fMRI'
#resting_indices = [i for i, r in enumerate(rec_type) if (r == 'Resting_State_fMRI' and np.shape(collected_signals[i])[0] > 100 and np.shape(collected_signals[i])[1] == 121)]
resting_indices = [i for i, r in enumerate(rec_type) if (np.shape(collected_signals[i])[0] > 100 and np.shape(collected_signals[i])[1] == 121)]

# Filter identifiers, patient_ID, and state_ID accordingly
identifiers_resting = [identifiers[i] for i in resting_indices]
patient_ID_resting = [identifiers[i][1] for i in resting_indices]
state_ID_resting = [identifiers[i][0] for i in resting_indices]
collected_signals_sel = [collected_signals[i] for i in resting_indices]
                

In [ ]:
# Convert state_ID to numeric codes: 'AD' -> 0, 'CC' -> 1, 'MCI' -> 2
state_ID_numeric = []
for sid in state_ID_resting:
    if sid == 'CC':
        state_ID_numeric.append(0)
    elif sid == 'EMCI':
        state_ID_numeric.append(1)
    elif sid == 'MCI':
        state_ID_numeric.append(2)
    elif sid == 'LCMI':
        state_ID_numeric.append(3)
    elif sid == 'AD':
        state_ID_numeric.append(4)
    else:
        state_ID_numeric.append(-1)  # Unknown label

In [ ]:
state_ID_numeric

In [ ]:
# Select only classes 0 and 4
selected_classes = [0,  4]
filtered_indices = [i for i, c in enumerate(state_ID_numeric) if c in selected_classes]

# Apply the filter to all relevant lists
state_ID_numeric = [state_ID_numeric[i] for i in filtered_indices]
identifiers_resting = [identifiers_resting[i] for i in filtered_indices]
patient_ID_resting = [patient_ID_resting[i] for i in filtered_indices]
state_ID_resting = [state_ID_resting[i] for i in filtered_indices]
collected_signals_sel = [collected_signals_sel[i] for i in filtered_indices]


In [ ]:
# EVALUATING PCA
# Concatenate all signals from all subjects along the time axis
concatenated_signals_list = []
avg_activity = []
for sig in collected_signals_sel:
    #sig = np.copy(sig)[5:,:]
    #sig_centered = sig - np.mean(sig)
    #sig_centered = sig_centered - np.mean(sig_centered, axis=1, keepdims=True)
    #concatenated_signals_list.append(sig_centered.T)
    concatenated_signals_list.append(sig.T)
    avg_activity.append(np.mean(sig, axis=0))
    
concatenated_signals = np.concatenate(concatenated_signals_list, axis=1)


TIMES = concatenated_signals.shape[1]
data = np.copy(concatenated_signals) #
#data=values[ch,:].T

mean = np.mean(data, axis=1)
std_data = np.std(data, axis=1)

centered_data = (data - np.tile(mean,(TIMES,1) ).T)/np.tile(std_data,(TIMES,1) ).T

plt.plot(centered_data.T)
plt.xlabel('time')

plt.ylabel('BOLD')


plt.show()

In [ ]:
# Step 2: Compute the covariance matrix
covariance_matrix = np.cov(centered_data.T, rowvar=False)

# Step 3: Perform eigen decomposition
eigenvalues, eigenvectors = np.linalg.eig(covariance_matrix)
explained_variance_ratio = eigenvalues / np.sum(eigenvalues)

# Step 4: Sort eigenvalues and eigenvectors in descending order

sorted_indices_common = np.argsort(eigenvalues)[::-1]
sorted_eigenvalues_common = eigenvalues[sorted_indices_common]
sorted_eigenvectors_common = eigenvectors[:, sorted_indices_common]

# Step 5: Calculate the principal components
principal_components_common = np.dot(centered_data.T, sorted_eigenvectors_common)

plt.subplot(121)
plt.plot(explained_variance_ratio[sorted_indices_common],'.')
plt.xlabel('n pc')
plt.ylabel('explained variance')

plt.subplot(122)

plt.plot(np.cumsum(explained_variance_ratio[sorted_indices_common]),'.')
plt.xlabel('n pc')
plt.ylabel('cum sum explained variance')

plt.tight_layout()

#concatenated_signals = np.copy(principal_components[:50,:])

N_sites,TIMES = np.shape(concatenated_signals)
trial_duration = TIMES

# Step 2: Compute the covariance matrix
centered_data = np.copy(np.double(concatenated_signals)).T
#covariance_matrix = np.cov( np.double(centered_data) , rowvar=False

dt = 0.005

norm_mua_target_tot = np.copy(centered_data)
norm_mua_target = np.copy(centered_data)


In [ ]:
plt.figure(figsize=(10, 6))

# Plot the first 3 principal components
for i in range(3):
    plt.plot(principal_components_common[:1000, i], label=f'PC {i+1}')


plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('First 3 Principal Components of the Data')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def train_test_pinv(feedback_factor=10., sigma_inner_dynamics=0.):
    """Collect driven reservoir activity and target MUA.

    Always runs in driven mode (feedback = norm_mua_target[:, t]).
    Returns arrays of shape (T-1, N) and (T-1, N_sites).
    """
    T      = network_reservoire.T - 1
    N      = network_reservoire.N
    N_out  = network_reservoire.O
    scale  = feedback_factor            # mua_relative_weight

    res_act = np.empty((T, N),     dtype=np.float64)
    mua_act = np.empty((T, N_out), dtype=np.float64)

    network_reservoire.acc = 0

    for t in range(T):
        inp = scale * norm_mua_target[:, t]   # already a new array (scalar * slice)
        network_reservoire.step_rate(inp, sigma_dyn=sigma_inner_dynamics)
        res_act[t] = network_reservoire.X
        mua_act[t] = norm_mua_target[:, t]

    print(res_act.shape, mua_act.shape)
    return res_act, mua_act


In [ ]:
def train_test(n_Rep, if_collect=True, sigma_noise_dyn=0,
               sigma_inner_dynamics=0, feedback_factor=10.,
               if_plot_results=False, if_use_reservoir=True,
               perturbation=None):
    """Run n_Rep epochs of driven-then-autonomous simulation.

    Returns (mean_ERROR, mean_ERROR_BEHAVIOR, mean_ERROR_PCA,
             mean_ERROR_BEHAVIOR_NET, mean_ERROR_S_PCA).
    """
    mua_relative_weight = feedback_factor
    add_noise           = sigma_noise_dyn > 0.
    T_run               = trial_duration - 1
    N_out               = network_reservoire.O

    # Constants that do not change across steps/epochs
    driven_steps = min(5, T_run)   # t = 0..5 are driven (if > 0)

    ERROR       = np.empty(n_Rep)
    ERROR_BEH   = np.empty(n_Rep)
    ERROR_PCA   = np.empty(n_Rep)
    ERROR_S_PCA = np.empty(n_Rep)
    ERROR_BEH_NET = np.empty(n_Rep)

    for epoch in range(n_Rep):
        network_reservoire.reset()

        if if_collect:
            INP          = np.empty((T_run, N_out),                dtype=np.float64)
            S_res        = np.empty((T_run, network_reservoire.N), dtype=np.float64)
            S_fromres    = np.empty((T_run, N_out),                dtype=np.float64)
            BEHAVIOR_ACC = np.zeros(T_run,                         dtype=np.float64)
            PCA          = np.empty((T_run, N_out),                dtype=np.float64)

        # â”€â”€ Phase 1: driven (t = 0 â€¦ driven_steps) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        for t in range(driven_steps):
            inp = mua_relative_weight * norm_mua_target[:, t]
            network_reservoire.step_rate(inp, sigma_dyn=sigma_inner_dynamics)
            if add_noise:
                network_reservoire.y += np.random.normal(0, sigma_noise_dyn, N_out)
            if perturbation:
                network_reservoire.y = ((1 - network_reservoire.alpha) * network_reservoire.y
                                        + network_reservoire.alpha
                                        * (network_reservoire.J_targ @ network_reservoire.X))
            if if_collect:
                INP[t]       = inp
                S_res[t]     = network_reservoire.X
                S_fromres[t] = network_reservoire.y
                PCA[t]       = network_reservoire.y

        # â”€â”€ Phase 2: autonomous (t = driven_steps â€¦ T_run) â”€â”€â”€â”€â”€â”€â”€â”€â”€
        for t in range(driven_steps, T_run):
            inp = mua_relative_weight * network_reservoire.y
            network_reservoire.step_rate(inp, sigma_dyn=sigma_inner_dynamics)
            if add_noise:
                network_reservoire.y += np.random.normal(0, sigma_noise_dyn, N_out)
            if perturbation:
                network_reservoire.y = ((1 - network_reservoire.alpha) * network_reservoire.y
                                        + network_reservoire.alpha
                                        * (network_reservoire.J_targ @ network_reservoire.X))
            if if_collect:
                INP[t]       = inp
                S_res[t]     = network_reservoire.X
                S_fromres[t] = network_reservoire.y
                PCA[t]       = network_reservoire.y

        ERROR[epoch]       = 0.
        ERROR_BEH[epoch]   = 0.
        ERROR_PCA[epoch]   = 0.
        ERROR_S_PCA[epoch] = 0.
        ERROR_BEH_NET[epoch] = 0.

    # â”€â”€ Optional plot â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if if_plot_results:
        data_target = norm_mua_target.T
        principal_components_data = np.dot(data_target, sorted_eigenvectors_common)

        centered = S_fromres
        principal_components = np.dot(centered, sorted_eigenvectors_common)

        fig, axs = plt.subplots(2, 3, figsize=(10, 5))
        axs[0, 0].imshow(INP[:, :15].T, aspect='auto', cmap='copper',
                         extent=[0, T_run * dt, 0, 15])
        axs[0, 0].set_title('input')
        axs[0, 2].set_title('output')

        for k, ax in enumerate(axs[1]):
            ax.plot(np.arange(len(principal_components_data)) * dt,
                    principal_components_data[:, k], label='data')
            ax.plot(np.arange(len(principal_components)) * dt,
                    principal_components[:, k], label='model')
            ax.set_xlabel('time (s)')
            ax.set_ylabel(f'PC {k + 1}')
            ax.legend()

        plt.tight_layout()
        plt.savefig('fig.eps')
        plt.savefig('fig.png')
        plt.show()

    # â”€â”€ Store results on network object â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if if_use_reservoir:
        data = S_fromres
    else:
        data = S_fromres   # fallback (S was unused in original)

    principal_components      = np.dot(data, sorted_eigenvectors_common)
    principal_components_data = np.dot(norm_mua_target.T, sorted_eigenvectors_common)

    network_reservoire.data                      = data
    network_reservoire.principal_components      = principal_components
    network_reservoire.principal_components_data = principal_components_data

    return (np.mean(ERROR), np.mean(ERROR_BEH), np.mean(ERROR_PCA),
            np.mean(ERROR_BEH_NET), np.mean(ERROR_S_PCA))


In [ ]:
spectral_radius = .95

In [ ]:
import gym
import numpy as np
from copy import deepcopy
from tqdm import trange
from matplotlib import pyplot as plt
from matplotlib.animation import ArtistAnimation
from gym.envs.mujoco import AntEnv  # Assuming you're using MuJoCo's Ant environment


from scipy.stats import pearsonr
import pickle

#network_reservoire.J = (network_reservoire.J + network_reservoire.J.T)/2./np.sqrt(2.)

if_save_res = True
if_load_res = False
N_PC = 20 # Number of principal components to use
N_sites = 121 # Number of sites in the Schaefer atlas

trial_duration = 139

N, I, O, TIME = 2000, N_sites, N_sites , 600
shape = (N, I, O, TIME)

dt = .005# / T;

tau_m_f = .0001 * dt
tau_m_s = .0001 * dt
sigma_input = .01

n_electrodes = N_sites
n_pca = n_electrodes

# Here we build the dictionary of the simulation parameters
par = {'tau_m_f' : tau_m_f,'tau_m_s' : tau_m_s,
    'N' : N, 'T' : TIME, 'dt' : dt,
    'sigma_input' : sigma_input, 'shape' : shape}
n_pca_feedback = N_sites#3#only

shape = ( N , n_pca_feedback, N_sites, TIME)
par['shape']=shape

network_reservoire = RESERVOIRE_SIMPLE(par)

if if_save_res:
    with open('network_reservoire.pkl', 'wb') as f:
        pickle.dump(network_reservoire, f)

if if_load_res:
    with open('network_reservoire.pkl', 'rb') as f:
        network_reservoire = pickle.load(f)

network_reservoire.J = network_reservoire.J * spectral_radius

eigenvalues, eigenvectors = np.linalg.eig(network_reservoire.J)

real_parts = np.real(eigenvalues)
imaginary_parts = np.imag(eigenvalues)

# Create a scatter plot
plt.scatter(real_parts, imaginary_parts)

theta = np.linspace(0, 2 * np.pi, 100)

# Circle center coordinates
center = (0, 0)
# Circle radius
radius = 1.

# Calculate circle points
x = center[0] + radius * np.cos(theta)
y = center[1] + radius * np.sin(theta)

# Plot the circle
plt.plot(x, y)

err_collected = []


In [ ]:
# Initialize storage for fitted weights
fitted_weights = {}

trial_duration = 139

fitted_weights_list = []
FC_SIM_COLLECTED = []
X_coll = []
Y_coll = []

means = []

all_weights = []

detrended_means = []
FC_degree_collected = []

FC_plus_collected_sel = []
FC_plus_collected_sel_sim = []

FC_collected = []

Y_coll_sim = []

# Perform 10 trials per dataset
for patient in range(len(concatenated_signals_list)):

    data = np.copy(concatenated_signals_list[patient]) # Use the first run of the patient data
    #data = data - np.mean(data, axis=1, keepdims=True) 
    network_reservoire.T = np.shape(data)[1]  # Set the trial duration based on the data shape
    network_reservoire.reset()  # Reset the reservoir network for each patient

    #norm_mua_target = np.copy(data)
    # Remove PC1 as detrending
    u, s, vt = np.linalg.svd(data - np.mean(data, axis=1, keepdims=True), full_matrices=False)
    pc1 = np.outer(u[:, 0], s[0] * vt[0, :])
    norm_mua_target = np.copy(data)# - pc1)
    # Project onto first 5 principal components only
    #norm_mua_target = np.dot(norm_mua_target.T, sorted_eigenvectors[:, :N_PC]).T
    detrended_means.append(np.mean(norm_mua_target, axis=1, keepdims=True))

    eigenvectors_5 = sorted_eigenvectors_common[:, 0:100]
    
    # Step 1: project onto the first 5 PCs
    pc_5_scores = np.dot(data.T, eigenvectors_5)
    # Step 2: reconstruct (reproject back)
    norm_mua_target = np.dot(pc_5_scores, eigenvectors_5.T).T

    # Z-score normalization
    #norm_mua_target = (norm_mua_target - np.mean(norm_mua_target, axis=1, keepdims=True)) / np.std(norm_mua_target, axis=1, keepdims=True)
    #norm_mua_target = (norm_mua_target - np.mean(norm_mua_target, axis=1, keepdims=True)) / np.std(norm_mua_target, axis=1, keepdims=True)
    fc_matrix = np.corrcoef(norm_mua_target)
    fc_matrix = np.nan_to_num(fc_matrix)   
    FC_collected.append(fc_matrix.flatten())
    FC_degree_collected.append(np.sum(np.corrcoef(norm_mua_target),axis=1))

    fc_matrix_delayed = np.corrcoef(norm_mua_target.T[:-1,:].T, norm_mua_target.T[1:,:].T)
    fc_matrix_delayed = np.nan_to_num(fc_matrix_delayed)
    fc_delayed_flat = fc_matrix_delayed[121:,:121]

    # Concatenate both flattened arrays
    fc_features = np.concatenate([fc_matrix.flatten(), fc_delayed_flat.flatten()])

    print(np.shape(fc_features))

    FC_plus_collected_sel.append(fc_features)
    
    means.append(np.mean(data, axis=1, keepdims=True))
    weights_list = []
    

    X, Y = train_test_pinv(feedback_factor=recurrent_factor)

    X = np.array(X)[10:,:]
    Y = np.array(Y)[10:,:]
    
    Y_coll.append(Y)
    X_coll.append(X)
    weights_list_mean=[]
    
    for trial in range(1):
        # Train and test with noise realization

        noise_size = .025*.5
        
        penalty = 0.01  # Regularization penalty
        W_out = np.linalg.pinv(X + np.random.normal(0, noise_size, size=np.shape(X))).dot(Y)
        #W_out -= penalty * np.sign(W_out)  # Apply L1 penalty to reduce large weights
        mse = np.mean((Y - np.dot(X, W_out))**2)
        print("mse:", mse)
        weights_list_mean.append(W_out)
    
    weight_mean = 0

    for elements in weights_list_mean:
        weight_mean += elements

    weight_mean /= len(weights_list_mean)
    #W_out = np.mean(np.array(weights_list_mean),axis=0)
    weights_list.append(weight_mean)
    all_weights.append(weight_mean.flatten())
    #all_weights.append( np.sum(weight_mean,axis=1))
    # Store weights for the current dataset
    fitted_weights[patient] = weights_list
    fitted_weights_list.append(W_out)

    network_reservoire.Jout = np.copy(W_out.T)
    err,err_out,err_pca,_,err_spca = train_test(1,sigma_noise_dyn=.0,feedback_factor=recurrent_factor,if_plot_results=False)

    fc_sim_data = np.corrcoef(network_reservoire.data.T)
    fc_sim_data = np.nan_to_num(fc_sim_data)
    FC_SIM_COLLECTED.append(fc_sim_data.flatten())

    Y_coll_sim.append(network_reservoire.data.T)

    fc_matrix_delayed_sim = np.corrcoef(network_reservoire.data[:-1,:].T, network_reservoire.data[1:,:].T)
    fc_matrix_delayed_sim = np.nan_to_num(fc_matrix_delayed_sim)
    fc_delayed_flat = fc_matrix_delayed_sim[121:,:121]

    # Concatenate both flattened arrays
    fc_features = np.concatenate([fc_sim_data.flatten(), fc_delayed_flat.flatten()])

    FC_plus_collected_sel_sim.append(fc_features)

    print(np.corrcoef(FC_SIM_COLLECTED[-1],FC_collected[-1])[0,1])  


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import matplotlib.pyplot as plt

# =========================
# Prepare tensors
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"

X_tensors = [torch.tensor(x, dtype=torch.float32).to(device) for x in X_coll]  # (t,N)
Y_tensors = [torch.tensor(y, dtype=torch.float32).to(device) for y in Y_coll]  # (t,k)

num_patients = len(X_tensors)
t, N = X_tensors[0].shape
_, k = Y_tensors[0].shape

M = 10   # number of shared basis features

# =========================
# Train/Test split
# =========================


from sklearn.model_selection import train_test_split

# =========================
# Train/test split
# =========================
unique_patients = np.unique(np.array(patient_ID_resting))
train_patients, test_patients = train_test_split(
    unique_patients, test_size=0.1, shuffle=True
)

patient_indices_arr = np.array(patient_ID_resting)

# Map patient IDs to model indices
train_ids = [i for i, pid in enumerate(patient_indices_arr) if pid in set(train_patients)]
test_ids  = [i for i, pid in enumerate(patient_indices_arr) if pid in set(test_patients)]
# =========================
# Model
# =========================

class BasisReadout(nn.Module):
    def __init__(self, M, k, N, num_patients):
        super().__init__()
        self.W = nn.Parameter(torch.randn(M, k, N)*0.05)
        self.g = nn.Parameter(torch.randn(num_patients, M)*0.05)

    def forward(self, X, pid):
        W_eff = torch.einsum("m,mkn->kn", self.g[pid], self.W)
        return X @ W_eff.T

model = BasisReadout(M, k, N, num_patients).to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# =========================
# Training
# =========================

train_curve = []
test_curve  = []



In [ ]:
import torch
import torch.optim as optim
import torch.linalg as LA
import matplotlib.pyplot as plt
from tqdm import trange

# ============================================================
# Regularization helpers
# ============================================================

def archetype_ortho_loss(W, lam=1e-2):
    """
    Penalize off-diagonal elements of the Gram matrix of archetypes.
    Encourages archetypes to be mutually orthogonal / distinct.
    W: [M, K, N]
    """
    W_flat = W.reshape(W.shape[0], -1)                          # [M, K*N]
    W_norm = W_flat / (W_flat.norm(dim=1, keepdim=True) + 1e-8) # normalize rows
    G = W_norm @ W_norm.T                                        # [M, M] Gram matrix
    off_diag = G - torch.eye(G.shape[0], device=W.device)
    return lam * (off_diag ** 2).sum()


def archetype_norm_loss(W, lam=1e-3):
    """
    Encourage each archetype to have unit norm.
    Prevents degenerate scaling solutions.
    """
    W_flat = W.reshape(W.shape[0], -1)
    norms = W_flat.norm(dim=1)
    return lam * ((norms - 1.0) ** 2).sum()


def g_sparsity_loss(g_soft, lam=1e-3):
    """
    Entropy-based sparsity on mixing coefficients.
    High entropy  → diffuse  (penalized)
    Low entropy   → sparse   (preferred)
    """
    entropy = -(g_soft * torch.log(g_soft + 1e-8)).sum(-1).mean()
    return lam * entropy


def get_g_soft(g):
    """Simplex projection via softmax. Works with raw g from model.g."""
    return torch.softmax(g, dim=-1)


# ============================================================
# Training loop
# ============================================================

# --- Regularization weights (tune these) ---
LAM_ORTHO   = 0*1e-2   # archetype orthogonality
LAM_NORM    = 0*1e-3   # archetype unit-norm
LAM_SPARSE  = 0*1e-3   # g entropy sparsity
N_ADAPT     = 50     # optimisation steps for test-time g adaptation
LR_ADAPT    = 5e-2   # learning rate for test-time g

train_curve, test_curve = [], []

for epoch in trange(1000):

    model.train()
    train_loss = 0

    for pid in train_ids:
        opt.zero_grad()

        # Forward pass – model internally uses softmax on g logits
        pred = model(X_tensors[pid], pid)
        
        loss = loss_fn(pred, Y_tensors[pid])

        # ── Regularization ──────────────────────────────────
        loss += archetype_ortho_loss(model.W)
        loss += archetype_norm_loss(model.W)

        # Sparsity on this patient's mixing weights
        g_soft = get_g_soft(model.g[pid])
        loss += g_sparsity_loss(g_soft)
        # ────────────────────────────────────────────────────

        loss.backward()
        opt.step()
        train_loss += loss.item()

    train_loss /= len(train_ids)

    # ----------------------------------------------------------
    if epoch % 100 == 0:

        model.eval()
        W_fixed = model.W.detach()

        adapt_errors = []
        test_g = []

        # Warm-start: mean of train g vectors
        mean_g = model.g.detach()[train_ids].mean(0)

        for pid in test_ids:

            # Start from the mean archetype mixture of training subjects
            g_logit = mean_g.clone().requires_grad_(True)
            opt_g   = optim.Adam([g_logit], lr=LR_ADAPT)

            X = X_tensors[pid]
            Y = Y_tensors[pid]

            best_loss = float('inf')
            best_g    = None

            for _ in range(N_ADAPT):
                opt_g.zero_grad()

                # Enforce simplex constraint via softmax
                g_soft = get_g_soft(g_logit)
                W_eff  = torch.einsum("m,mkn->kn", g_soft, W_fixed)
                pred   = X @ W_eff.T
                loss   = loss_fn(pred, Y)
                loss  += g_sparsity_loss(g_soft)    # keep sparsity at test time too

                loss.backward()
                opt_g.step()

                # Track best, not just last step
                pred_loss = loss_fn(pred.detach(), Y).item()
                if pred_loss < best_loss:
                    best_loss = pred_loss
                    best_g    = get_g_soft(g_logit.detach().cpu())

            adapt_errors.append(best_loss)
            test_g.append(best_g)

        test_g   = torch.stack(test_g)
        test_loss = sum(adapt_errors) / len(adapt_errors)

        print(f"epoch: {epoch:5d} | train: {train_loss:.4f} | test: {test_loss:.4f}")

        train_curve.append(train_loss)
        test_curve.append(test_loss)

        # ── Diagnostics ──────────────────────────────────────
        train_g_soft = get_g_soft(model.g.detach()[train_ids]).cpu()

        # g entropy: low = sparse (good), high = diffuse (collapsed)
        all_g = torch.cat([train_g_soft, test_g], dim=0)
        g_entropy = -(all_g * torch.log(all_g + 1e-8)).sum(-1).mean()

        # Archetype similarity matrix (for orthogonality check)
        W_flat = W_fixed.reshape(W_fixed.shape[0], -1)
        W_norm = W_flat / (W_flat.norm(dim=1, keepdim=True) + 1e-8)
        gram   = (W_norm @ W_norm.T).cpu()

        # ── PCA of g ─────────────────────────────────────────
        all_g_centered = all_g - all_g.mean(0, keepdim=True)
        U, S, V = LA.svd(all_g_centered, full_matrices=False)
        proj = all_g_centered @ V[:2].T

        train_proj = proj[:len(train_g_soft)]
        test_proj  = proj[len(train_g_soft):]

        # ── Mean archetype importance ─────────────────────────
        # all_g: [n_subjects, M] — average weight each archetype gets
        mean_importance      = all_g.mean(0).numpy()        # [M]
        std_importance       = all_g.std(0).numpy()         # [M]
        train_importance     = train_g_soft.mean(0).numpy()
        test_importance      = test_g.mean(0).numpy()
        M                    = len(mean_importance)
        archetype_labels     = [f"A{m}" for m in range(M)]

        # ── Plotting ─────────────────────────────────────────
        plt.clf()
        fig, ax = plt.subplots(1, 4, figsize=(18, 4))

        # Panel 0: loss curves
        ax[0].plot(train_curve, label="train")
        ax[0].plot(test_curve,  label="test")
        ax[0].set_yscale("log")
        ax[0].set_title("Loss")
        ax[0].legend()

        # Panel 1: latent g PCA
        train_colors = [state_ID_numeric[i] for i in train_ids]
        test_colors  = [state_ID_numeric[i] for i in test_ids]

        ax[1].scatter(train_proj[:, 0], train_proj[:, 1],
                      c=train_colors, marker="o", label="train")
        ax[1].scatter(test_proj[:, 0],  test_proj[:, 1],
                      c=test_colors,  marker="^", label="test")
        ax[1].set_title(f"Latent g PCA  (entropy={g_entropy:.3f})")
        ax[1].legend()

        # Panel 2: archetype Gram matrix
        im_gram = ax[2].imshow(gram.numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
        ax[2].set_title("Archetype Gram matrix\n(diagonal=1 ideal)")
        ax[2].set_xlabel("Archetype")
        ax[2].set_ylabel("Archetype")
        ax[2].set_xticks(range(M))
        ax[2].set_yticks(range(M))
        plt.colorbar(im_gram, ax=ax[2], fraction=0.046, pad=0.04)

        # Panel 3: mean archetype importance (bar chart)
        x = range(M)
        ax[3].bar(x, train_importance, width=0.3, align="center",
                  label="train", alpha=0.8)
        ax[3].bar([i + 0.32 for i in x], test_importance, width=0.3,
                  align="center", label="test", alpha=0.8)
        ax[3].errorbar([i + 0.16 for i in x], mean_importance,
                       yerr=std_importance, fmt="none",
                       color="black", capsize=4, linewidth=1.2,
                       label="±std (all)")
        ax[3].set_xticks([i + 0.16 for i in x])
        ax[3].set_xticklabels(archetype_labels)
        ax[3].set_ylim(0, 1)
        ax[3].set_ylabel("Mean g weight")
        ax[3].set_title("Archetype importance")
        ax[3].axhline(1 / M, color="grey", linestyle="--",
                      linewidth=1, label=f"uniform (1/{M})")
        ax[3].legend(fontsize=7)

        plt.tight_layout()
        plt.savefig(f"figures_MC_twoclasses/arch_train_{epoch:05d}.png", dpi=72, bbox_inches="tight"); plt.close("all")

In [ ]:
import torch
import torch.optim as optim
import torch.linalg as LA
import matplotlib.pyplot as plt
from tqdm import trange

# ============================================================
# Regularization helpers
# ============================================================

def archetype_ortho_loss(W, lam=1e-2):
    W_flat = W.reshape(W.shape[0], -1)
    W_norm = W_flat / (W_flat.norm(dim=1, keepdim=True) + 1e-8)
    G = W_norm @ W_norm.T
    off_diag = G - torch.eye(G.shape[0], device=W.device)
    return lam * (off_diag ** 2).sum()


def archetype_norm_loss(W, lam=1e-3):
    W_flat = W.reshape(W.shape[0], -1)
    norms = W_flat.norm(dim=1)
    return lam * ((norms - 1.0) ** 2).sum()


def g_sparsity_loss(g, lam=1e-3):
    """L1-based sparsity on raw g (no softmax needed)."""
    return lam * g.abs().sum()


# ============================================================
# Hyperparameters
# ============================================================

LAM_ORTHO  = 0 * 1e-2
LAM_NORM   = 0 * 1e-3
LAM_SPARSE = 0 * 1e-3
N_ADAPT    = 50
LR_ADAPT   = 5e-2

train_curve, test_curve = [], []

# ============================================================
# Training loop
# ============================================================

for epoch in trange(1000):

    model.train()
    train_loss = 0

    for pid in train_ids:
        opt.zero_grad()

        pred = model(X_tensors[pid], pid)
        loss = loss_fn(pred, Y_tensors[pid])

        loss += archetype_ortho_loss(model.W, lam=LAM_ORTHO)
        loss += archetype_norm_loss(model.W, lam=LAM_NORM)

        # Sparsity directly on raw g (no softmax)
        loss += g_sparsity_loss(model.g[pid], lam=LAM_SPARSE)

        loss.backward()
        opt.step()
        train_loss += loss.item()

    train_loss /= len(train_ids)

    # ----------------------------------------------------------
    if epoch % 100 == 0:

        model.eval()
        W_fixed = model.W.detach()

        adapt_errors = []
        test_g = []

        # Warm-start from mean of train g vectors
        mean_g = model.g.detach()[train_ids].mean(0)

        for pid in test_ids:

            # Raw unconstrained g, no logits / softmax
            g = mean_g.clone().requires_grad_(True)
            opt_g = optim.Adam([g], lr=LR_ADAPT)

            X = X_tensors[pid]
            Y = Y_tensors[pid]

            best_loss = float('inf')
            best_g    = None

            for _ in range(N_ADAPT):
                opt_g.zero_grad()

                W_eff = torch.einsum("m,mkn->kn", g, W_fixed)
                pred  = X @ W_eff.T
                loss  = loss_fn(pred, Y)
                # No sparsity penalty here to avoid train/test mismatch

                loss.backward()
                opt_g.step()

                pred_loss = loss.item()
                if pred_loss < best_loss:
                    best_loss = pred_loss
                    best_g    = g.detach().cpu().clone()

            adapt_errors.append(best_loss)
            test_g.append(best_g)

        test_g    = torch.stack(test_g)
        test_loss = sum(adapt_errors) / len(adapt_errors)

        print(f"epoch: {epoch:5d} | train: {train_loss:.4f} | test: {test_loss:.4f}")

        train_curve.append(train_loss)
        test_curve.append(test_loss)

        # ── Diagnostics ──────────────────────────────────────
        train_g = model.g.detach().cpu()[train_ids]

        all_g     = torch.cat([train_g, test_g], dim=0)
        g_entropy = -(
            torch.softmax(all_g, dim=-1)
            * torch.log_softmax(all_g, dim=-1)
        ).sum(-1).mean()  # just for display; computed on softmax of raw g

        W_flat = W_fixed.reshape(W_fixed.shape[0], -1)
        W_norm = W_flat / (W_flat.norm(dim=1, keepdim=True) + 1e-8)
        gram   = (W_norm @ W_norm.T).cpu()

        # PCA of g
        all_g_centered = all_g - all_g.mean(0, keepdim=True)
        U, S, V = LA.svd(all_g_centered, full_matrices=False)
        proj = all_g_centered @ V[:2].T

        train_proj = proj[:len(train_g)]
        test_proj  = proj[len(train_g):]

        # Archetype importance (via abs value since g is unconstrained)
        g_abs            = all_g.abs()
        g_norm           = g_abs / (g_abs.sum(-1, keepdim=True) + 1e-8)
        mean_importance  = g_norm.mean(0).numpy()
        std_importance   = g_norm.std(0).numpy()
        train_importance = g_norm[:len(train_g)].mean(0).numpy()
        test_importance  = g_norm[len(train_g):].mean(0).numpy()
        M                = len(mean_importance)
        archetype_labels = [f"A{m}" for m in range(M)]

        # ── Plotting ─────────────────────────────────────────
        plt.clf()
        fig, ax = plt.subplots(1, 4, figsize=(18, 4))

        ax[0].plot(train_curve, label="train")
        ax[0].plot(test_curve,  label="test")
        ax[0].set_yscale("log")
        ax[0].set_title("Loss")
        ax[0].legend()

        train_colors = [state_ID_numeric[i] for i in train_ids]
        test_colors  = [state_ID_numeric[i] for i in test_ids]

        ax[1].scatter(train_proj[:, 0], train_proj[:, 1],
                      c=train_colors, marker="o", label="train")
        ax[1].scatter(test_proj[:, 0],  test_proj[:, 1],
                      c=test_colors,  marker="^", label="test")
        ax[1].set_title(f"Latent g PCA  (entropy={g_entropy:.3f})")
        ax[1].legend()

        im_gram = ax[2].imshow(gram.numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
        ax[2].set_title("Archetype Gram matrix\n(diagonal=1 ideal)")
        ax[2].set_xlabel("Archetype")
        ax[2].set_ylabel("Archetype")
        ax[2].set_xticks(range(M))
        ax[2].set_yticks(range(M))
        plt.colorbar(im_gram, ax=ax[2], fraction=0.046, pad=0.04)

        x = range(M)
        ax[3].bar(x, train_importance, width=0.3, align="center",
                  label="train", alpha=0.8)
        ax[3].bar([i + 0.32 for i in x], test_importance, width=0.3,
                  align="center", label="test", alpha=0.8)
        ax[3].errorbar([i + 0.16 for i in x], mean_importance,
                       yerr=std_importance, fmt="none",
                       color="black", capsize=4, linewidth=1.2,
                       label="±std (all)")
        ax[3].set_xticks([i + 0.16 for i in x])
        ax[3].set_xticklabels(archetype_labels)
        ax[3].set_ylim(0, 1)
        ax[3].set_ylabel("Mean |g| weight (normalized)")
        ax[3].set_title("Archetype importance")
        ax[3].axhline(1 / M, color="grey", linestyle="--",
                      linewidth=1, label=f"uniform (1/{M})")
        ax[3].legend(fontsize=7)

        plt.tight_layout()
        plt.savefig(f"figures_MC_twoclasses/arch_train_{epoch:05d}.png", dpi=72, bbox_inches="tight"); plt.close("all")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import zscore
from tqdm import tqdm

# ============================================================
# Step 1 – Evaluate G for ALL subjects via pseudoinverse
# ============================================================

model.eval()
W_fixed  = model.W.detach()

all_pids  = list(range(len(X_coll)))   # every subject id
all_g_emb = []                    # will hold one g vector per subject

print("Computing g for all subjects via pseudoinverse...")

# Stack archetypes as columns: [M, K*N] → each row is a flattened archetype
W_flat = W_fixed.reshape(W_fixed.shape[0], -1)   # [M, K*N]
W_pinv = torch.linalg.pinv(W_flat.T)             # [M, K*N]  pinv of [K*N, M]

for pid in tqdm(all_pids):

    X = X_coll[pid]
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X, dtype=torch.float32)

    Y = Y_tensors[pid]
    if not isinstance(Y, torch.Tensor):
        Y = torch.tensor(Y, dtype=torch.float32)

    with torch.no_grad():
        # Step 1: recover effective weight matrix via pinv(X) @ Y → [N, K]
        W_eff = torch.linalg.pinv(X) @ Y          # [N, K]
        W_eff_flat = W_eff.T.reshape(-1)           # [K*N]  matches W_flat layout

        # Step 2: solve g = pinv(W_stacked.T) @ vec(W_eff)
        g = W_pinv @ W_eff_flat                    # [M]

        # Project onto simplex (softmax to keep comparable scale)
        g_soft = torch.softmax(g, dim=-1).cpu()

    all_g_emb.append(g_soft.numpy())

all_g_emb = np.stack(all_g_emb)   # [N_subjects, M]

print(f"G matrix collected: {all_g_emb.shape}")

# ============================================================
# Step 2 – Classification with Random Forest
# ============================================================

n_pc = all_g_emb.shape[1]   # use up to M dimensions

# Labels: one per subject, aligned with all_pids
all_colors_arr = np.array([state_ID_numeric[pid] for pid in all_pids])

# --- Outlier removal via z-score ---
z_scores    = zscore(all_g_emb, axis=0)
non_outliers = (np.abs(z_scores) < 20_000_000).all(axis=1)

data_cleaned     = all_g_emb[non_outliers]
filtered_labels  = all_colors_arr[non_outliers]
patient_ids_clean = np.array(all_pids)[non_outliers]

# --- Class balancing (MDD=0 vs Control=4, adjust if needed) ---
def balance_classes(X, y, class0=0, class1=4, seed=None):
    rng = np.random.default_rng(seed)
    idx0 = np.where(y == class0)[0]
    idx1 = np.where(y == class1)[0]
    n    = min(len(idx0), len(idx1))
    rng.shuffle(idx0)
    rng.shuffle(idx1)
    sel  = np.concatenate([idx0[:n], idx1[:n]])
    return X[sel], y[sel]

# --- Repeated RF across random seeds ---
test_accuracies_all     = []
train_accuracies_all    = []
shuffled_test_all       = []

for random_state in range(10):
    print(f"Seed {random_state}")
    rng_seed = np.random.randint(1000)

    test_accs, train_accs, shuffled_accs = [], [], []

    for k in range(1, n_pc + 1):

        # Patient-level split (no data leakage)
        unique_patients = np.unique(patient_ids_clean)
        train_patients, test_patients = train_test_split(
            unique_patients, test_size=0.2,
            random_state=rng_seed, shuffle=True
        )

        train_mask = np.isin(patient_ids_clean, train_patients)
        test_mask  = np.isin(patient_ids_clean, test_patients)

        X_train = data_cleaned[train_mask, :k]
        X_test  = data_cleaned[test_mask,  :k]
        y_train = filtered_labels[train_mask]
        y_test  = filtered_labels[test_mask]

        X_train, y_train = balance_classes(X_train, y_train, seed=rng_seed)
        X_test,  y_test  = balance_classes(X_test,  y_test,  seed=rng_seed)

        rf = RandomForestClassifier(
            n_estimators=300,
            max_depth=4,
            min_samples_split=2,
            min_samples_leaf=5,
            random_state=rng_seed,
            n_jobs=-1,
        )
        rf.fit(X_train, y_train)

        y_pred_train = rf.predict(X_train)
        y_pred_test  = rf.predict(X_test)

        train_accs.append(accuracy_score(y_train, y_pred_train))
        test_accs.append(accuracy_score(y_test,  y_pred_test))

        shuffled_accs.append(
            accuracy_score(np.random.permutation(y_test), y_pred_test)
        )

    test_accuracies_all.append(test_accs)
    train_accuracies_all.append(train_accs)
    shuffled_test_all.append(shuffled_accs)

# ============================================================
# Step 3 – Aggregate & Plot RF accuracy
# ============================================================

ks = range(1, n_pc + 1)

avg_test   = np.median(test_accuracies_all,  axis=0)
avg_train  = np.median(train_accuracies_all, axis=0)
avg_shuff  = np.median(shuffled_test_all,    axis=0)

p20_test,  p80_test  = np.percentile(test_accuracies_all,  [20, 80], axis=0)
p20_train, p80_train = np.percentile(train_accuracies_all, [20, 80], axis=0)
p20_shuff, p80_shuff = np.percentile(shuffled_test_all,    [20, 80], axis=0)

# ============================================================
# Step 4 – 2D PCA of g embeddings coloured by class
# ============================================================

g_tensor = torch.tensor(data_cleaned, dtype=torch.float32)
g_centered = g_tensor - g_tensor.mean(0, keepdim=True)
_, _, V = torch.linalg.svd(g_centered, full_matrices=False)
proj2d = (g_centered @ V[:2].T).numpy()   # [N, 2]

unique_classes = np.unique(filtered_labels)
cmap = plt.colormaps["tab10"].resampled(len(unique_classes))
class_colors = {c: cmap(i) for i, c in enumerate(unique_classes)}

fig_pca, ax_pca = plt.subplots(figsize=(5, 4))
for cls in unique_classes:
    mask = filtered_labels == cls
    ax_pca.scatter(
        proj2d[mask, 0], proj2d[mask, 1],
        c=[class_colors[cls]],
        label=str(cls),
        alpha=0.7, s=25, edgecolors='none'
    )
ax_pca.set_xlabel("PC 1")
ax_pca.set_ylabel("PC 2")
ax_pca.set_title("2D PCA of archetype embeddings g")
legend = ax_pca.legend(title="Class", frameon=False)
ax_pca.spines['top'].set_visible(False)
ax_pca.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig_g_pca.png', dpi=150)
plt.savefig('fig_g_pca.pdf')
plt.show()

# ============================================================
# Step 5 – RF accuracy plot
# ============================================================

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(ks, avg_test,  '-o', label='Test accuracy')
ax.fill_between(ks, p20_test,  p80_test,  alpha=0.2)

ax.plot(ks, avg_train, '-o', label='Train accuracy')
ax.fill_between(ks, p20_train, p80_train, alpha=0.2)

ax.plot(ks, avg_shuff, '-o', label='Shuffled (chance)')
ax.fill_between(ks, p20_shuff, p80_shuff, alpha=0.2)

ax.axhline(0.5, color='grey', linestyle=':', linewidth=1, label='50% baseline')

ax.set_xlabel('Number of archetype dimensions (k)')
ax.set_ylabel('Accuracy')
ax.set_title(
    f'RF classification from archetype g\n'
    f'Max test accuracy: {np.max(avg_test):.2f}  '
    f'(at k={np.argmax(avg_test)+1})'
)

legend = ax.legend()
legend.get_frame().set_facecolor('none')
legend.get_frame().set_edgecolor('none')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('fig_lda.png', dpi=150)
plt.savefig('fig_lda.pdf')
plt.show()

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Step 2 – Classification with LDA
# ============================================================

n_pc = all_g_emb.shape[1]

all_colors_arr = np.array([state_ID_numeric[pid] for pid in all_pids])

z_scores     = zscore(all_g_emb, axis=0)
non_outliers = (np.abs(z_scores) < 20_000_000).all(axis=1)

data_cleaned      = all_g_emb[non_outliers]
filtered_labels   = all_colors_arr[non_outliers]
patient_ids_clean = np.array(all_pids)[non_outliers]

def balance_classes(X, y, class0=0, class1=4, seed=None):
    rng  = np.random.default_rng(seed)
    idx0 = np.where(y == class0)[0]
    idx1 = np.where(y == class1)[0]
    n    = min(len(idx0), len(idx1))
    rng.shuffle(idx0); rng.shuffle(idx1)
    sel  = np.concatenate([idx0[:n], idx1[:n]])
    return X[sel], y[sel]

test_accuracies_all, train_accuracies_all, shuffled_test_all = [], [], []

for random_state in range(10):
    print(f"Seed {random_state}")
    rng_seed = np.random.randint(1000)

    test_accs, train_accs, shuffled_accs = [], [], []

    for k in range(1, n_pc + 1):

        unique_patients = np.unique(patient_ids_clean)
        train_patients, test_patients = train_test_split(
            unique_patients, test_size=0.2,
            random_state=rng_seed, shuffle=True
        )

        train_mask = np.isin(patient_ids_clean, train_patients)
        test_mask  = np.isin(patient_ids_clean, test_patients)

        X_train, y_train = balance_classes(data_cleaned[train_mask, :k], filtered_labels[train_mask], seed=rng_seed)
        X_test,  y_test  = balance_classes(data_cleaned[test_mask,  :k], filtered_labels[test_mask],  seed=rng_seed)

        lda = LinearDiscriminantAnalysis()
        lda.fit(X_train, y_train)

        train_accs.append(accuracy_score(y_train, lda.predict(X_train)))
        test_accs.append( accuracy_score(y_test,  lda.predict(X_test)))
        shuffled_accs.append(accuracy_score(np.random.permutation(y_test), lda.predict(X_test)))

    test_accuracies_all.append(test_accs)
    train_accuracies_all.append(train_accs)
    shuffled_test_all.append(shuffled_accs)

# ============================================================
# Step 3 – Aggregate & Plot
# ============================================================

ks = range(1, n_pc + 1)

avg_test,  avg_train,  avg_shuff  = [np.median(x, axis=0) for x in [test_accuracies_all, train_accuracies_all, shuffled_test_all]]
p20_test,  p80_test  = np.percentile(test_accuracies_all,  [20, 80], axis=0)
p20_train, p80_train = np.percentile(train_accuracies_all, [20, 80], axis=0)
p20_shuff, p80_shuff = np.percentile(shuffled_test_all,    [20, 80], axis=0)

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(ks, avg_test,  '-o', label='Test accuracy')
ax.fill_between(ks, p20_test,  p80_test,  alpha=0.2)
ax.plot(ks, avg_train, '-o', label='Train accuracy')
ax.fill_between(ks, p20_train, p80_train, alpha=0.2)
ax.plot(ks, avg_shuff, '-o', label='Shuffled (chance)')
ax.fill_between(ks, p20_shuff, p80_shuff, alpha=0.2)
ax.axhline(0.5, color='grey', linestyle=':', linewidth=1, label='50% baseline')

ax.set_xlabel('Number of archetype dimensions (k)')
ax.set_ylabel('Accuracy')
ax.set_title(
    f'LDA classification from archetype g\n'
    f'Max test accuracy: {np.max(avg_test):.2f}  (at k={np.argmax(avg_test)+1})'
)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('fig_lda.png', dpi=150)
plt.savefig('fig_lda.pdf')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# 1. Prepare Split
unique_patients = np.unique(patient_ids_clean)
train_p, test_p = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_mask = np.isin(patient_ids_clean, train_p)
test_mask = np.isin(patient_ids_clean, test_p)

X_train_raw, y_train = balance_classes(data_cleaned[train_mask], filtered_labels[train_mask], seed=42)
X_test_raw, y_test = balance_classes(data_cleaned[test_mask], filtered_labels[test_mask], seed=42)

# 2. Extract unit vectors for the axes
# Axis 1: PC1 (Direction of max variance)
pca = PCA(n_components=1).fit(X_train_raw)
v_pc1 = pca.components_[0] 

# Axis 2: LDA (Direction of max discrimination)
lda = LDA(n_components=1).fit(X_train_raw, y_train)
v_lda = lda.coef_[0]
v_lda /= np.linalg.norm(v_lda) # Normalize to unit length

# 3. Project data onto these two specific vectors
# Projection = dot product of data with the unit vector
def project_onto_axes(X, v1, v2):
    x_proj = X @ v1
    y_proj = X @ v2
    return np.vstack([x_proj, y_proj]).T

proj_train = project_onto_axes(X_train_raw, v_pc1, v_lda)
proj_test  = project_onto_axes(X_test_raw, v_pc1, v_lda)

# 4. Plotting
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
datasets = [(proj_train, y_train, "Train Set"), (proj_test, y_test, "Test Set")]

for i, (data, labels, title) in enumerate(datasets):
    ax = axes[i]
    for cls in np.unique(labels):
        mask = labels == cls
        ax.scatter(data[mask, 0], data[mask, 1], label=f"Class {cls}", alpha=0.6, edgecolors='w')
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("First Principal Component (Variance)")
    if i == 0: ax.set_ylabel("Discriminant Direction (LDA)")
    ax.axhline(0, color='grey', lw=1, ls='--')
    ax.axvline(0, color='grey', lw=1, ls='--')
    ax.legend(frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Calculate Redundancy (Cosine Similarity)
cos_sim = np.dot(v_pc1, v_lda)
print(f"Alignment between PC1 and LDA: {cos_sim:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# â”€â”€ Panel 0: overlapping histograms â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
bins = np.linspace(
    min(np.min(ccs_orig), np.min(ccs_reduced)),
    max(np.max(ccs_orig), np.max(ccs_reduced)),
    40
)

ax[0].hist(ccs_orig,    bins=bins, alpha=0.6, color='steelblue',
           label=f'original   (med={np.median(ccs_orig):.3f})')
ax[0].hist(ccs_reduced, bins=bins, alpha=0.6, color='orange',
           label=f'reduced    (med={np.median(ccs_reduced):.3f})')
ax[0].axvline(np.median(ccs_orig),    color='steelblue', linestyle='--', linewidth=1.5)
ax[0].axvline(np.median(ccs_reduced), color='orange',    linestyle='--', linewidth=1.5)
ax[0].set_xlabel("FC correlation"); ax[0].set_ylabel("Count")
ax[0].set_title("Original vs reduced")
ax[0].legend(frameon=False)

# â”€â”€ Panel 1: scatter original vs reduced (one dot per subject) â”€
ax[1].scatter(ccs_orig, ccs_reduced, alpha=0.6, s=20)
lims = [min(np.min(ccs_orig), np.min(ccs_reduced)),
        max(np.max(ccs_orig), np.max(ccs_reduced))]
ax[1].plot(lims, lims, 'r--', linewidth=1, label='y=x')
ax[1].set_xlabel("CC original"); ax[1].set_ylabel("CC reduced")
ax[1].set_title("Per-subject comparison")
ax[1].legend(frameon=False)

# â”€â”€ Stats â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
t_stat, p_val = stats.wilcoxon(ccs_orig, ccs_reduced)
diff = np.array(ccs_reduced) - np.array(ccs_orig)
print(f"Original  â€” mean: {np.mean(ccs_orig):.3f}  median: {np.median(ccs_orig):.3f}  std: {np.std(ccs_orig):.3f}")
print(f"Reduced   â€” mean: {np.mean(ccs_reduced):.3f}  median: {np.median(ccs_reduced):.3f}  std: {np.std(ccs_reduced):.3f}")
print(f"Difference (reduced - original) â€” mean: {diff.mean():.3f}  std: {diff.std():.3f}")
print(f"Wilcoxon signed-rank test: stat={t_stat:.3f}  p={p_val:.4f}")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_delayed_fc(data, delay):
    """Return delayed FC matrix for a given delay (data shape: regions x time)."""
    if delay < 0:
        raise ValueError("delay must be >= 0")
    if delay == 0:
        return np.nan_to_num(np.corrcoef(data))
    if data.shape[1] <= delay:
        raise ValueError("delay must be smaller than the number of time points")

    lead = data[:, :-delay]
    lag = data[:, delay:]
    corr = np.corrcoef(lead, lag)
    delayed_block = corr[data.shape[0]:, :data.shape[0]]
    return np.nan_to_num(delayed_block)


times_to_skip = 10
max_delay = 40
num_examples = 20  # first 20 patients

all_corr = []
all_corr_ref = []
all_corr_ref_top = []

# --- Collect all trajectories ---
for patinent in range(len(Y_coll)):

    empirical_data_trimmed = Y_coll[patinent][:, times_to_skip:].T
    sim_data_trimmed = Y_coll_sim[patinent][times_to_skip:, :]

    empirical_data_trimmed_even = empirical_data_trimmed[:, ::2]
    empirical_data_trimmed_odd = empirical_data_trimmed[:, 1::2]

    correlations_vs_delay = []
    corr_val_ref_vs_delay = []
    corr_val_ref_top_vs_delay = []
    valid_delays = []

    fc_emp_0 = compute_delayed_fc(empirical_data_trimmed, 0)

    for delay in range(0, max_delay + 1):
        if delay > 0 and (
            empirical_data_trimmed.shape[1] <= delay or
            sim_data_trimmed.shape[1] <= delay
        ):
            break

        fc_emp = compute_delayed_fc(empirical_data_trimmed, delay)
        fc_sim = compute_delayed_fc(sim_data_trimmed, delay)

        correlations_vs_delay.append(np.corrcoef(fc_emp.flatten(), fc_sim.flatten())[0, 1])
        corr_val_ref_vs_delay.append(np.corrcoef(fc_emp.flatten(), fc_emp_0.flatten())[0, 1])
        corr_val_ref_top_vs_delay.append(
            np.corrcoef(
                compute_delayed_fc(empirical_data_trimmed_even, delay).flatten(),
                compute_delayed_fc(empirical_data_trimmed_odd, delay).flatten()
            )[0, 1]
        )
        valid_delays.append(delay)

    all_corr.append(correlations_vs_delay)
    all_corr_ref.append(corr_val_ref_vs_delay)
    all_corr_ref_top.append(corr_val_ref_top_vs_delay)

# -----------------------------
# 1ï¸âƒ£ Plot 20 single panels
# -----------------------------
plt.figure(figsize=(20, 12))
for i, pat in enumerate(range(num_examples)):
    ax = plt.subplot(4, 5, i + 1)
    ax.plot(valid_delays, all_corr[pat], marker='o')
    ax.plot(valid_delays, all_corr_ref[pat], marker='s', linestyle='--', color='gray')
    ax.plot(np.array(valid_delays)*2, all_corr_ref_top[pat], marker='s', linestyle='--', color='k')

    ax.axhline(0, color='k', linestyle='--', linewidth=1)
    ax.set_ylim(-0.1, 1)
    ax.set_xlim(0, max_delay)
    ax.set_title(f'Patient {pat+1}', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)
    
    if i < 15: ax.set_xlabel('')
    if i % 5 != 0: ax.set_ylabel('')

plt.tight_layout()
plt.show()

# -----------------------------
# 2ï¸âƒ£ Plot chunk average (median + IQR)
# -----------------------------
def plot_with_stats(trajectories, color, label):
    arr = np.stack(trajectories)
    median = np.median(arr, axis=0)
    p25 = np.percentile(arr, 25, axis=0)
    p75 = np.percentile(arr, 75, axis=0)
    plt.plot(median, color=color, label=label, linewidth=2)
    plt.fill_between(np.arange(len(median)), p25, p75, color=color, alpha=0.2)

plt.figure(figsize=(5, 4))
plot_with_stats(all_corr, 'blue', 'Empirical vs Simulated')
plot_with_stats(all_corr_ref, 'gray', 'Empirical vs 0-lag')
plot_with_stats(all_corr_ref_top, 'k', 'Even vs Odd')

plt.xlabel('Delay (time steps)')
plt.ylabel('Pearson corr (delayed FC)')
plt.title('Delayed FC: Median Â± IQR across patients')
plt.ylim(-0.1, 1)
plt.xlim(0, max_delay)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_delayed_fc(data, delay):
    if delay == 0:
        return np.nan_to_num(np.corrcoef(data))
    lead = data[:, :-delay]
    lag  = data[:, delay:]
    corr = np.corrcoef(lead, lag)
    return np.nan_to_num(corr[data.shape[0]:, :data.shape[0]])


times_to_skip = 10
max_delay     = 40
num_examples  = 20
P             = len(Y_coll)

all_corr          = []
all_corr_ref      = []
all_corr_ref_top  = []

# ============================================================
# Cross-subject FC matrix at delay=0
# cc_matrix[i, j] = corr(FC_sim_i, FC_emp_j)
# Diagonal = same-subject match (should be maximum per row)
# ============================================================
cc_matrix = np.zeros((P, P))

fc_emp_flat_all = []   # store all empirical FC for cross-subject comparison
fc_sim_flat_all = []   # store all simulated FC

for patient in range(P):
    emp = Y_coll[patient][:, times_to_skip:].T         # [N, T]
    sim = Y_coll_sim[patient][times_to_skip:, :]     # [N, T]
    fc_emp_flat_all.append(compute_delayed_fc(emp, 0).flatten())
    fc_sim_flat_all.append(compute_delayed_fc(sim, 0).flatten())

# Fill cross-subject matrix
for i in range(P):
    for j in range(P):
        cc_matrix[i, j] = np.corrcoef(
            fc_sim_flat_all[i],
            fc_emp_flat_all[j]
        )[0, 1]

# ============================================================
# Plot 3 â€” cross-subject FC correlation matrix  (delay=0)
# Diagonal should be the maximum of each row
# ============================================================
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

im = ax[0].imshow(cc_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax[0].set_title("CC matrix: sim_i vs emp_j\n(diagonal = same subject)")
ax[0].set_xlabel("Empirical subject j")
ax[0].set_ylabel("Simulated subject i")
plt.colorbar(im, ax=ax[0])

# Diagonal vs off-diagonal distributions
diag_vals    = np.diag(cc_matrix)
offdiag_vals = cc_matrix[~np.eye(P, dtype=bool)]
ax[1].hist(offdiag_vals, bins=40, alpha=0.6, color='gray',  label='off-diagonal')
ax[1].hist(diag_vals,    bins=20, alpha=0.8, color='blue',  label='diagonal (same subj)')
ax[1].axvline(np.median(diag_vals),    color='blue', linestyle='--',
              label=f'diag median={np.median(diag_vals):.2f}')
ax[1].axvline(np.median(offdiag_vals), color='gray', linestyle='--',
              label=f'off-diag median={np.median(offdiag_vals):.2f}')
ax[1].set_xlabel("Pearson correlation"); ax[1].set_ylabel("Count")
ax[1].set_title("Same-subject vs cross-subject FC match")
ax[1].legend(fontsize=8)


ax[2].axvline(1, color='red', linestyle='--', label='rank=1 (perfect)')
ax[2].set_xlabel("Rank of same-subject match")
ax[2].set_ylabel("Count")

ax[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nSame-subject FC corr  â€” median: {np.median(diag_vals):.3f}  "
      f"mean: {np.mean(diag_vals):.3f}")
print(f"Cross-subject FC corr â€” median: {np.median(offdiag_vals):.3f}  "
      f"mean: {np.mean(offdiag_vals):.3f}")
